# Gas de Ehrenfest

O modelo do Gás de Ehrenfest descreve um sistema com as seguintes regras:

1. O sistema consiste em dois compartimentos (caixas ou urnas), designados como Compartimento 1 e Compartimento 2.

2. Existem N partículas no total, numeradas de 1 a N.

3. Inicialmente, algumas dessas partículas estão localizadas no Compartimento 1, enquanto o restante está no Compartimento 2.

4. Em cada unidade de tempo, uma partícula é escolhida aleatoriamente  e transferida para o compartimento oposto.

5. Este procedimento é repetido sequencialmente ao longo do tempo.

In [ ]:
# ============================================================
# SIMULAÇÃO ANIMADA DO MODELO DO GÁS DE EHRENNFEST
# Código pronto para Google Colab
# ============================================================

!pip -q install pillow

import numpy as np
import matplotlib.pyplot as plt

from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.patches import FancyBboxPatch, Rectangle
from IPython.display import Image, display
from google.colab import files


# ============================================================
# PARÂMETROS DA SIMULAÇÃO
# ============================================================

N = 120                    # número total de partículas
N_A_INICIAL = 100          # partículas inicialmente na câmara A
NUMERO_DE_PASSOS = 600     # número de passos de Monte Carlo

NUMERO_DE_FRAMES = 120     # quantidade de quadros do GIF
INTERVALO_MS = 80          # intervalo entre os quadros
SEMENTE = 15               # semente aleatória

NOME_DO_GIF = "modelo_ehrenfest.gif"


# ============================================================
# VALIDAÇÃO DOS PARÂMETROS
# ============================================================

if N <= 0:
    raise ValueError("N deve ser maior que zero.")

if not 0 <= N_A_INICIAL <= N:
    raise ValueError("N_A_INICIAL deve estar entre 0 e N.")

if NUMERO_DE_PASSOS <= 0:
    raise ValueError("NUMERO_DE_PASSOS deve ser maior que zero.")


# ============================================================
# SIMULAÇÃO DO MODELO DE EHRENFEST
# ============================================================

rng = np.random.default_rng(SEMENTE)

# True representa uma partícula na câmara A.
# False representa uma partícula na câmara B.
particulas_na_camara_A = np.zeros(N, dtype=bool)

particulas_na_camara_A[:N_A_INICIAL] = True

# Embaralha os índices das partículas
rng.shuffle(particulas_na_camara_A)

# Armazena o estado do sistema a cada passo
estados = [particulas_na_camara_A.copy()]

# Armazena a quantidade de partículas na câmara A
historico_n_A = [int(particulas_na_camara_A.sum())]


for passo in range(NUMERO_DE_PASSOS):

    # Escolhe aleatoriamente uma das partículas
    indice_escolhido = int(rng.integers(0, N))

    # A partícula escolhida atravessa para a outra câmara
    particulas_na_camara_A[indice_escolhido] = (
        not particulas_na_camara_A[indice_escolhido]
    )

    # Salva o novo estado
    estados.append(particulas_na_camara_A.copy())

    # Registra a quantidade de partículas na câmara A
    historico_n_A.append(
        int(particulas_na_camara_A.sum())
    )


historico_n_A = np.asarray(historico_n_A)


# ============================================================
# FUNÇÃO PARA GERAR AS POSIÇÕES DAS PARTÍCULAS
# ============================================================

def gerar_coordenadas(quantidade, caixa, rng_local, margem=0.25):
    """
    Gera posições aleatórias dentro de uma região retangular.

    caixa = (posição_x, posição_y, largura, altura)
    """

    x, y, largura, altura = caixa

    coordenadas_x = rng_local.uniform(
        x + margem,
        x + largura - margem,
        quantidade
    )

    coordenadas_y = rng_local.uniform(
        y + margem,
        y + altura - margem,
        quantidade
    )

    return coordenadas_x, coordenadas_y


# ============================================================
# FUNÇÃO PARA DESENHAR AS DUAS CÂMARAS
# ============================================================

def desenhar_recipiente(ax):

    # Câmara A
    camara_A = FancyBboxPatch(
        (0.4, 1.2),
        4.2,
        3.5,
        boxstyle="round,pad=0.08,rounding_size=0.9",
        linewidth=1.8,
        facecolor="lightgray",
        edgecolor="black",
        zorder=1
    )

    # Câmara B
    camara_B = FancyBboxPatch(
        (6.4, 1.2),
        4.2,
        3.5,
        boxstyle="round,pad=0.08,rounding_size=0.9",
        linewidth=1.8,
        facecolor="lightgray",
        edgecolor="black",
        zorder=1
    )

    # Canal de ligação
    canal = Rectangle(
        (4.45, 2.55),
        2.1,
        0.8,
        linewidth=1.8,
        facecolor="lightgray",
        edgecolor="black",
        zorder=0
    )

    ax.add_patch(canal)
    ax.add_patch(camara_A)
    ax.add_patch(camara_B)

    ax.text(
        2.5,
        5.05,
        "Câmara A",
        fontsize=16,
        ha="center"
    )

    ax.text(
        8.5,
        5.05,
        "Câmara B",
        fontsize=16,
        ha="center"
    )

    ax.text(
        5.5,
        2.95,
        "canal",
        fontsize=10,
        ha="center",
        va="center"
    )

    ax.set_xlim(0, 11)
    ax.set_ylim(0.6, 5.7)

    ax.set_aspect("equal")
    ax.axis("off")


# ============================================================
# SELEÇÃO DOS PASSOS QUE APARECERÃO NO GIF
# ============================================================

quantidade_de_frames = min(
    NUMERO_DE_FRAMES,
    len(estados)
)

passos_dos_frames = np.unique(
    np.linspace(
        0,
        len(estados) - 1,
        quantidade_de_frames,
        dtype=int
    )
)


# ============================================================
# CRIAÇÃO DA FIGURA
# ============================================================

figura = plt.figure(figsize=(12, 8))

grade = figura.add_gridspec(
    2,
    1,
    height_ratios=[3.2, 1.5],
    hspace=0.25
)

ax_gas = figura.add_subplot(grade[0])
ax_grafico = figura.add_subplot(grade[1])

desenhar_recipiente(ax_gas)


# Objetos que representam as partículas
particulas_A = ax_gas.scatter(
    [],
    [],
    s=55,
    facecolor="tomato",
    edgecolor="black",
    linewidth=0.8,
    zorder=3
)

particulas_B = ax_gas.scatter(
    [],
    [],
    s=55,
    facecolor="tomato",
    edgecolor="black",
    linewidth=0.8,
    zorder=3
)


# Textos informativos
texto_A = ax_gas.text(
    0.7,
    4.55,
    "",
    fontsize=14
)

texto_B = ax_gas.text(
    10.3,
    4.55,
    "",
    fontsize=14,
    ha="right"
)

texto_passo = ax_gas.text(
    5.5,
    0.75,
    "",
    fontsize=12,
    ha="center"
)


# ============================================================
# CONFIGURAÇÃO DO GRÁFICO TEMPORAL
# ============================================================

ax_grafico.set_xlim(
    0,
    NUMERO_DE_PASSOS
)

ax_grafico.set_ylim(
    0,
    N
)

ax_grafico.set_xlabel(
    "Passos de tempo"
)

ax_grafico.set_ylabel(
    r"$n_A(t)$"
)

ax_grafico.set_title(
    "Evolução do número de partículas na câmara A"
)

# Linha correspondente ao valor médio de equilíbrio
ax_grafico.axhline(
    N / 2,
    color="black",
    linestyle="--",
    linewidth=1.3,
    label=rf"Equilíbrio médio: $N/2={N/2:.0f}$"
)

linha_historico, = ax_grafico.plot(
    [],
    [],
    color="royalblue",
    linewidth=2,
    label=r"$n_A(t)$"
)

ax_grafico.grid(
    alpha=0.25
)

ax_grafico.legend(
    loc="upper right"
)

figura.suptitle(
    "Simulação do modelo do gás de Ehrenfest",
    fontsize=18
)


# ============================================================
# FUNÇÃO DE ATUALIZAÇÃO DA ANIMAÇÃO
# ============================================================

def atualizar_animacao(numero_do_frame):

    passo_atual = int(
        passos_dos_frames[numero_do_frame]
    )

    estado_atual = estados[passo_atual]

    n_A = int(
        estado_atual.sum()
    )

    n_B = N - n_A

    # Semente própria de cada quadro
    rng_frame = np.random.default_rng(
        SEMENTE + 10000 + passo_atual
    )

    # Coordenadas das partículas da câmara A
    x_A, y_A = gerar_coordenadas(
        n_A,
        caixa=(0.4, 1.2, 4.2, 3.5),
        rng_local=rng_frame
    )

    # Coordenadas das partículas da câmara B
    x_B, y_B = gerar_coordenadas(
        n_B,
        caixa=(6.4, 1.2, 4.2, 3.5),
        rng_local=rng_frame
    )

    if n_A > 0:
        particulas_A.set_offsets(
            np.column_stack((x_A, y_A))
        )
    else:
        particulas_A.set_offsets(
            np.empty((0, 2))
        )

    if n_B > 0:
        particulas_B.set_offsets(
            np.column_stack((x_B, y_B))
        )
    else:
        particulas_B.set_offsets(
            np.empty((0, 2))
        )

    texto_A.set_text(
        rf"$n_A(t)={n_A}$"
    )

    texto_B.set_text(
        rf"$n_B(t)=N-n_A={n_B}$"
    )

    texto_passo.set_text(
        f"Passos de tempo: {passo_atual}"
    )

    # Atualização do gráfico
    eixo_x = np.arange(
        passo_atual + 1
    )

    eixo_y = historico_n_A[
        :passo_atual + 1
    ]

    linha_historico.set_data(
        eixo_x,
        eixo_y
    )

    return (
        particulas_A,
        particulas_B,
        texto_A,
        texto_B,
        texto_passo,
        linha_historico
    )


# ============================================================
# CRIAÇÃO DA ANIMAÇÃO
# ============================================================

animacao = FuncAnimation(
    figura,
    atualizar_animacao,
    frames=len(passos_dos_frames),
    interval=INTERVALO_MS,
    blit=False,
    repeat=True
)


# ============================================================
# SALVAR COMO GIF
# ============================================================

quadros_por_segundo = max(
    4,
    int(1000 / INTERVALO_MS)
)

animacao.save(
    NOME_DO_GIF,
    writer=PillowWriter(
        fps=quadros_por_segundo
    ),
    dpi=110
)

plt.close(figura)


# ============================================================
# RESULTADOS
# ============================================================

n_A_final = historico_n_A[-1]
n_B_final = N - n_A_final

print("Simulação concluída com sucesso!")
print(f"GIF criado: {NOME_DO_GIF}")
print(f"Número total de partículas: {N}")
print(f"Estado inicial: n_A = {N_A_INICIAL}")
print(f"Estado final: n_A = {n_A_final} e n_B = {n_B_final}")


# ============================================================
# EXIBIR O GIF NO GOOGLE COLAB
# ============================================================

display(
    Image(
        filename=NOME_DO_GIF
    )
)


![gas_ehrenfest.gif](https://raw.githubusercontent.com/marquinholol1/projeto_marcos_entropia/main/imagens/gas_ehrenfest.gif)

![ehrenfest.gif](https://raw.githubusercontent.com/marquinholol1/projeto_marcos_entropia/main/imagens/ehrenfest.gif)

## Modelo de Ehrenfest-Adaptado

Fonte: Atarés, Lorena, et al. "Helping pregraduate students reach deep understanding of the second law of thermodynamics." Education Sciences 11.9 (2021): 539.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
from matplotlib import rcParams

# ------------------------------------------------------------
# CONFIGURAÇÕES GERAIS
# ------------------------------------------------------------

# Fonte principal
rcParams["font.family"] = "DejaVu Sans"

# Mantém as partículas sempre nas mesmas posições
rng = np.random.default_rng(7)

# Criação da figura
fig = plt.figure(figsize=(13.5, 8.2), dpi=150)

# Eixo ocupando toda a figura
ax = fig.add_axes([0, 0, 1, 1])

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis("off")

# ------------------------------------------------------------
# CORES
# ------------------------------------------------------------

azul = "#4472C4"
fundo_macro = "#D9E1F2"
fundo_micro = "#EEF1F7"
fundo_caixas = "#F7F7F7"
cinza_comprimido = "#B3B3B3"
cor_borda = "#111111"

# ------------------------------------------------------------
# DIMENSÕES DA ESTRUTURA PRINCIPAL
# ------------------------------------------------------------

x_inicial = 0.075
x_final = 0.93

y_superior = 0.96
y_inferior = 0.17

altura_cabecalho = 0.085
altura_macro = 0.28

# ------------------------------------------------------------
# CABEÇALHO
# ------------------------------------------------------------

ax.add_patch(
    Rectangle(
        (x_inicial, y_superior - altura_cabecalho),
        x_final - x_inicial,
        altura_cabecalho,
        facecolor=azul,
        edgecolor="none"
    )
)

ax.text(
    (x_inicial + x_final) / 2,
    y_superior - altura_cabecalho / 2,
    "System: gas in a box",
    color="white",
    fontsize=15,
    ha="center",
    va="center"
)

# ------------------------------------------------------------
# FUNDOS DAS ÁREAS MACRO E MICRO
# ------------------------------------------------------------

y_inicio_macro = y_superior - altura_cabecalho - altura_macro

# Área do nível macroscópico
ax.add_patch(
    Rectangle(
        (x_inicial, y_inicio_macro),
        x_final - x_inicial,
        altura_macro,
        facecolor=fundo_macro,
        edgecolor="none"
    )
)

# Área do nível microscópico
ax.add_patch(
    Rectangle(
        (x_inicial, y_inferior),
        x_final - x_inicial,
        y_inicio_macro - y_inferior,
        facecolor=fundo_micro,
        edgecolor="none"
    )
)

# ------------------------------------------------------------
# TEXTOS LATERAIS
# ------------------------------------------------------------

ax.text(
    x_inicial + 0.038,
    y_inicio_macro + altura_macro * 0.53,
    r"$\it{Macro}$ level:"
    "\nwhat can actually"
    "\nbe detected"
    "\nmacroscopically",
    fontsize=15,
    ha="left",
    va="center",
    linespacing=1.25
)

ax.text(
    x_inicial + 0.025,
    y_inferior + (y_inicio_macro - y_inferior) * 0.56,
    r"$\it{Micro}$ level:"
    "\nsome specific"
    "\nmicroscopic"
    "\nconfigurations"
    "\n(microstates) of"
    "\neach macrostate",
    fontsize=15,
    ha="left",
    va="center",
    linespacing=1.25
)

# ------------------------------------------------------------
# MACROESTADOS
# ------------------------------------------------------------

largura_macro = 0.135
altura_caixa_macro = 0.12

x_macroestado_1 = 0.39
x_macroestado_2 = 0.71
y_macroestado = y_inicio_macro + 0.12

# Caixas dos dois macroestados
for x in [x_macroestado_1, x_macroestado_2]:
    ax.add_patch(
        Rectangle(
            (x, y_macroestado),
            largura_macro,
            altura_caixa_macro,
            facecolor="#E5E5E5",
            edgecolor=cor_borda,
            linewidth=2
        )
    )

# Parte ocupada pelo gás no macroestado comprimido
ax.add_patch(
    Rectangle(
        (x_macroestado_1, y_macroestado),
        largura_macro * 0.5,
        altura_caixa_macro,
        facecolor=cinza_comprimido,
        edgecolor="none"
    )
)

# Textos dos macroestados
ax.text(
    x_macroestado_1 + largura_macro / 2,
    y_macroestado - 0.02,
    "Macrostate 1:\ncompressed",
    fontsize=15,
    ha="center",
    va="top"
)

ax.text(
    x_macroestado_2 + largura_macro / 2,
    y_macroestado - 0.02,
    "Macrostate 2:\ndispersed",
    fontsize=15,
    ha="center",
    va="top"
)

# ------------------------------------------------------------
# CONFIGURAÇÃO DOS QUADROS DOS MICROESTADOS
# ------------------------------------------------------------

largura_painel = 0.14
altura_painel = 0.14

espaco_horizontal = 0.01
espaco_vertical = 0.008

x_grupo_comprimido = 0.31
x_grupo_disperso = 0.63

y_linha_superior = 0.41
y_linha_inferior = (
    y_linha_superior - altura_painel - espaco_vertical
)

# ------------------------------------------------------------
# FUNÇÕES AUXILIARES
# ------------------------------------------------------------

def desenhar_painel(x, y, pontos=None, grade=None):
    """
    Desenha uma caixa de microestado e coloca partículas dentro dela.

    grade:
        Tupla no formato (colunas, linhas, fração_horizontal).
    """

    ax.add_patch(
        Rectangle(
            (x, y),
            largura_painel,
            altura_painel,
            facecolor=fundo_caixas,
            edgecolor=cor_borda,
            linewidth=1.5
        )
    )

    # Partículas organizadas em grade
    if grade is not None:
        numero_colunas, numero_linhas, fracao_horizontal = grade

        valores_x = np.linspace(
            x + 0.012,
            x + largura_painel * fracao_horizontal - 0.012,
            numero_colunas
        )

        valores_y = np.linspace(
            y + 0.012,
            y + altura_painel - 0.012,
            numero_linhas
        )

        matriz_x, matriz_y = np.meshgrid(valores_x, valores_y)

        ax.scatter(
            matriz_x.ravel(),
            matriz_y.ravel(),
            s=10,
            c="black",
            zorder=5
        )

    # Partículas aleatórias
    elif pontos is not None:
        ax.scatter(
            pontos[:, 0],
            pontos[:, 1],
            s=9,
            c="black",
            zorder=5
        )


def criar_particulas(x, y, quantidade=52, concentradas=False):
    """
    Gera partículas dentro de um painel.

    concentradas=True:
        concentra as partículas no lado esquerdo.

    concentradas=False:
        distribui as partículas por toda a caixa.
    """

    if concentradas:
        # Distribuição beta produz maior concentração à esquerda
        valores_x = (
            x
            + 0.015
            + rng.beta(2.0, 5.0, quantidade)
            * (largura_painel * 0.48 - 0.025)
        )

    else:
        valores_x = (
            x
            + 0.012
            + rng.random(quantidade)
            * (largura_painel - 0.024)
        )

    valores_y = (
        y
        + 0.012
        + rng.random(quantidade)
        * (altura_painel - 0.024)
    )

    return np.column_stack((valores_x, valores_y))


# ------------------------------------------------------------
# MICROESTADOS DO MACROESTADO COMPRIMIDO
# ------------------------------------------------------------

# Superior esquerdo: partículas organizadas e comprimidas
desenhar_painel(
    x_grupo_comprimido,
    y_linha_superior,
    grade=(7, 8, 0.5)
)

# Superior direito
desenhar_painel(
    x_grupo_comprimido + largura_painel + espaco_horizontal,
    y_linha_superior,
    pontos=criar_particulas(
        x_grupo_comprimido + largura_painel + espaco_horizontal,
        y_linha_superior,
        quantidade=55,
        concentradas=True
    )
)

# Inferior esquerdo
desenhar_painel(
    x_grupo_comprimido,
    y_linha_inferior,
    pontos=criar_particulas(
        x_grupo_comprimido,
        y_linha_inferior,
        quantidade=52,
        concentradas=True
    )
)

# Inferior direito
desenhar_painel(
    x_grupo_comprimido + largura_painel + espaco_horizontal,
    y_linha_inferior,
    pontos=criar_particulas(
        x_grupo_comprimido + largura_painel + espaco_horizontal,
        y_linha_inferior,
        quantidade=55,
        concentradas=True
    )
)

# ------------------------------------------------------------
# MICROESTADOS DO MACROESTADO DISPERSO
# ------------------------------------------------------------

# Superior esquerdo: partículas organizadas em toda a caixa
desenhar_painel(
    x_grupo_disperso,
    y_linha_superior,
    grade=(7, 6, 1.0)
)

# Superior direito
desenhar_painel(
    x_grupo_disperso + largura_painel + espaco_horizontal,
    y_linha_superior,
    pontos=criar_particulas(
        x_grupo_disperso + largura_painel + espaco_horizontal,
        y_linha_superior,
        quantidade=48,
        concentradas=False
    )
)

# Inferior esquerdo
desenhar_painel(
    x_grupo_disperso,
    y_linha_inferior,
    pontos=criar_particulas(
        x_grupo_disperso,
        y_linha_inferior,
        quantidade=50,
        concentradas=False
    )
)

# Inferior direito
desenhar_painel(
    x_grupo_disperso + largura_painel + espaco_horizontal,
    y_linha_inferior,
    pontos=criar_particulas(
        x_grupo_disperso + largura_painel + espaco_horizontal,
        y_linha_inferior,
        quantidade=52,
        concentradas=False
    )
)

# ------------------------------------------------------------
# LEGENDA DA FIGURA
# ------------------------------------------------------------

y_legenda = 0.125

ax.text(
    x_inicial,
    y_legenda,
    "Figure 3.",
    fontsize=14,
    fontweight="bold",
    fontfamily="DejaVu Serif",
    ha="left",
    va="top"
)

texto_legenda = (
    "Different microstates of two (macro)states.  "
    "Macrostate 1 has low entropy, whereas\n"
    "macrostate 2 has the highest entropy.  "
    "In both cases, some microstates could be described as more\n"
    r"$\it{ordered}$, which does not imply any entropy variations "
    r"as compared to $\it{disordered}$ microstates."
)

ax.text(
    x_inicial + 0.11,
    y_legenda,
    texto_legenda,
    fontsize=14,
    fontfamily="DejaVu Serif",
    ha="left",
    va="top",
    linespacing=1.35
)

# ------------------------------------------------------------
# SALVAR E MOSTRAR
# ------------------------------------------------------------

plt.savefig(
    "macroestados_microestados.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

![tabela_comparativa_ehrenfest.png](https://raw.githubusercontent.com/marquinholol1/projeto_marcos_entropia/main/imagens/tabela_comparativa_ehrenfest.png)